# Probe a π0 checkpoint — `lang_ratio` scorecard (standalone)

Run in a **second kernel on the training pod** while the 50k run trains (~10 GB VRAM —
fits beside batch-32 training on an 80 GB card). Loads any `best.pt` from disk, rebuilds
the exact training wrapper, reconstructs the exact stratified val split from the
checkpoint's own seed/config, and runs the language-grounding probe.

**The scorecard per checkpoint:** `lang_ratio` (>2 = instructions steer the policy;
epoch-2 baseline 1.05) · gripper-at-grasp (300-frame window covers real grasps) ·
boundary-jump spot check (baseline 6.7°). Run it every epoch (~5–6 h) as `best.pt`
updates — same `CKPT_DIR`, just re-run all cells.

## 1 · Parameters

In [ ]:
CKPT_DIR   = "/workspace/checkpoints_pi0_v2_50k"   # which run's best.pt to probe
DATA_DIR   = "/workspace/hf_dataset"                # the converted dataset on this pod
PROBE_EPISODES = 3     # val episodes from distinct canonical tasks
PROBE_FRAMES   = 300   # covers the first grasp window (~frame 255) so the gripper
                       # metric is meaningful — the 120-frame probe reads 0 by construction

TASK_TEXT = ""         # dataset-class fallback only; never used (all episodes carry tasks)
print("params set")

## 2 · Dataset class (verbatim from the training notebook)

In [ ]:
import json, random
from pathlib import Path

import cv2
import numpy as np
import pyarrow.parquet as pq
import torch
from torch.utils.data import Dataset
from torchvision import transforms

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]


def _build_transform(image_size, aug_level):
    h, w = image_size
    if aug_level == "none":
        return transforms.Compose([
            transforms.Resize(image_size),
            transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)])
    if aug_level == "crops":   # UMI-calibrated jitter
        return transforms.Compose([
            transforms.Resize((int(h * 1.12), int(w * 1.12))),
            transforms.RandomCrop(image_size),
            transforms.ColorJitter(brightness=0.3, contrast=0.4, saturation=0.5, hue=0.08),
            transforms.RandomGrayscale(p=0.05),
            transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)])
    raise ValueError(f"unknown aug_level {aug_level!r}")


class FR5Dataset(Dataset):
    """LeRobot-v3 FR5 episodes -> (state, action chunk, pad mask, image, task)."""

    def __init__(self, root, chunk_size=50, image_size=(224, 224),
                 episode_indices=None, aug_level="none", frame_stride=1):
        self.root, self.chunk_size, self.image_size = Path(root), chunk_size, image_size
        self.frame_stride = max(1, int(frame_stride))
        self.info = json.loads((self.root / "meta/info.json").read_text())
        self.camera_keys = [k for k in self.info.get("features", {})
                            if k.startswith("observation.images.")] or                            ["observation.images.wrist_cam"]
        self.df = pq.read_table(self.root / "data/chunk-000/file-000.parquet").to_pandas()
        self.episodes = pq.read_table(
            self.root / "meta/episodes/chunk-000/file-000.parquet").to_pandas()
        if episode_indices is not None:
            self.episodes = self.episodes[
                self.episodes["episode_index"].isin(episode_indices)].reset_index(drop=True)
        self._samples = [(int(e.episode_index), t)
                         for _, e in self.episodes.iterrows()
                         for t in range(int(e.dataset_from_index), int(e.dataset_to_index), self.frame_stride)]
        tasks = self.root / "meta/tasks.parquet"
        self._task_map = (dict(zip(*pq.read_table(tasks).to_pandas()
                                   [["task_index", "task"]].T.values.tolist()))
                          if tasks.exists() else {})
        self._tf = _build_transform(image_size, aug_level)

    def __len__(self): return len(self._samples)

    def __getitem__(self, idx):
        ep_idx, frame_abs = self._samples[idx]
        row = self.df.iloc[frame_abs]
        ep = self.episodes[self.episodes.episode_index == ep_idx].iloc[0]
        ep_to = int(ep.dataset_to_index)

        state = torch.tensor(row["observation.state"], dtype=torch.float32)
        chunk = self.df.iloc[frame_abs:min(frame_abs + self.chunk_size, ep_to)]
        actions = torch.tensor(np.array(chunk["action"].tolist()), dtype=torch.float32)
        pad = self.chunk_size - len(actions)
        is_pad = torch.zeros(self.chunk_size, dtype=torch.bool)
        if pad > 0:
            actions = torch.cat([actions, actions[-1:].expand(pad, -1)])
            is_pad[-pad:] = True

        sample = {"observation.state": state, "action": actions, "action_is_pad": is_pad,
                  "task": self._task_map.get(int(row.get("task_index", 0)), TASK_TEXT)}
        for cam in self.camera_keys:                   # load EVERY camera (wrist + scene)
            jpg = self.root / "frames" / cam / f"ep-{ep_idx:03d}" / f"{int(row.frame_index):06d}.jpg"
            frame = cv2.imread(str(jpg))
            if frame is None:                          # video fallback
                cap = cv2.VideoCapture(str(self.root / "videos" / cam / "chunk-000" /
                                           f"file-{ep_idx:03d}.mp4"))
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame_index))
                ok, frame = cap.read(); cap.release()
                assert ok, f"missing {cam} frame ep{ep_idx} idx{int(row.frame_index)}"
            img = torch.from_numpy(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                                   ).permute(2, 0, 1).float() / 255.0
            sample[cam] = self._tf(img)
        return sample

    def get_stats(self):
        sub = self.df[self.df.episode_index.isin(self.episodes.episode_index.tolist())]
        s, a = (np.array(sub[k].tolist()) for k in ("observation.state", "action"))
        return {"state_mean": s.mean(0).astype(np.float32),
                "state_std":  s.std(0).clip(1e-6).astype(np.float32),
                "state_min":  s.min(0).astype(np.float32),
                "state_max":  s.max(0).astype(np.float32),
                "action_mean": a.mean(0).astype(np.float32),
                "action_std":  a.std(0).clip(1e-6).astype(np.float32),
                "action_min":  a.min(0).astype(np.float32),
                "action_max":  a.max(0).astype(np.float32)}

    @staticmethod
    def episode_split(n_episodes, val_frac=0.1, seed=42):
        idx = list(range(n_episodes))
        random.seed(seed); random.shuffle(idx)
        n_val = max(1, int(val_frac * n_episodes))
        return idx[n_val:], idx[:n_val]

print("FR5Dataset defined")

## 3 · Policy wrapper (verbatim — includes the lerobot compat patches)

In [ ]:
from contextlib import contextmanager
from dataclasses import dataclass

import torch
import torch.nn as nn
from transformers import AutoTokenizer

# lerobot eagerly imports its groot policy in policies/__init__, whose config has
# a dataclass bug (GR00TN15Config) that crashes the whole import on some builds.
# We don't use groot -> stub it in sys.modules before importing any lerobot policy.
import sys as _sys, types as _types
for _m in ("lerobot.policies.groot", "lerobot.policies.groot.configuration_groot"):
    _sys.modules.setdefault(_m, _types.ModuleType(_m))
_sys.modules["lerobot.policies.groot.configuration_groot"].GrootConfig = None

from lerobot.policies.pi0.configuration_pi0 import PI0Config as _LRConfig
from lerobot.policies.pi0.modeling_pi0 import PI0Policy

# lerobot's pi0-family passes a Long attention mask to torch.where, which
# PyTorch >= 2.8 rejects (needs bool). Wrap the method to cast the mask first.
import lerobot.policies.pi0.modeling_pi0 as _pimod
if not getattr(_pimod.PI0Pytorch._prepare_attention_masks_4d, '_bool_patched', False):
    _orig_mask4d = _pimod.PI0Pytorch._prepare_attention_masks_4d
    def _mask4d_bool(self, att_2d_masks, *a, **k):
        return _orig_mask4d(self, att_2d_masks.bool(), *a, **k)
    _mask4d_bool._bool_patched = True
    _pimod.PI0Pytorch._prepare_attention_masks_4d = _mask4d_bool
# lerobot 0.5.1's pi_gemma.py calls transformers' create_causal_mask(..., cache_position=...),
# but transformers >= 5.x dropped that parameter (no **kwargs to absorb it). Only the INFERENCE
# path hits it (select_action -> sample_actions), so training runs fine and it crashes at eval.
# Wrap the function to drop any kwargs the installed signature doesn't accept -> version-proof.
import inspect as _inspect, functools as _functools
import lerobot.policies.pi_gemma as _pg
if not getattr(_pg.create_causal_mask, "_kwarg_filtered", False):
    _ccm = _pg.create_causal_mask
    _ccm_names = {p.name for p in _inspect.signature(_ccm).parameters.values()}
    _ccm_varkw = any(p.kind == p.VAR_KEYWORD for p in _inspect.signature(_ccm).parameters.values())
    if not (_ccm_varkw or "cache_position" in _ccm_names):
        @_functools.wraps(_ccm)
        def _ccm_wrapped(*_a, **_k):
            return _ccm(*_a, **{_kk: _vv for _kk, _vv in _k.items() if _kk in _ccm_names})
        _ccm_wrapped._kwarg_filtered = True
        _pg.create_causal_mask = _ccm_wrapped
from lerobot.configs.types import PolicyFeature, FeatureType, NormalizationMode

STATE_KEY, IMAGE_KEY, ACTION_KEY = ("observation.state",
                                    "observation.images.wrist_cam", "action")
LANG_TOKENS    = "observation.language.tokens"
LANG_ATTN_MASK = "observation.language.attention_mask"
_IMN_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMN_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def mask_state(state, mode, rate, training):
    """Proprio modes: full = untouched · none = always zeroed ·
    dropout = per-sample zeroed with prob `rate` during training only."""
    if mode == "none":
        return torch.zeros_like(state)
    if mode == "dropout" and training:
        keep = (torch.rand(state.shape[0], 1, device=state.device) >= rate)
        return state * keep.to(state.dtype)
    return state


@dataclass
class PolicyCfg:
    state_dim:  int = 7
    action_dim: int = 7
    chunk_size: int = 50
    use_image:  bool = True
    num_inference_steps: int = 10
    max_state_dim:  int = 32
    max_action_dim: int = 32
    paligemma_variant:     str = "gemma_2b"
    action_expert_variant: str = "gemma_300m"
    tokenizer_max_length:  int = 48
    pretrained:            str = "lerobot/pi0_base"   # "" -> random init (smoke only)
    vlm_lora_rank:         int = 0                  # >0 -> LoRA on VLM q/k/v/o
    vlm_lora_alpha:        int = 32
    vlm_lora_dropout:      float = 0.05
    vlm_lora_targets:     tuple = ("q_proj", "k_proj", "v_proj", "o_proj")
    dtype:                  str  = "bfloat16"
    gradient_checkpointing: bool = True
    freeze_vision_encoder:  bool = False
    train_expert_only:      bool = False
    quantize:               str  = "none"   # none | nf4 | int8 (frozen VLM only)
    camera_names:         tuple = ("wrist_cam", "scene_cam")
    proprio_mode:         str   = "full"
    proprio_dropout_rate: float = 0.3


def _lerobot_config(cfg):
    feats = {STATE_KEY: PolicyFeature(type=FeatureType.STATE, shape=(cfg.state_dim,))}
    norm  = {"STATE": NormalizationMode.IDENTITY, "ACTION": NormalizationMode.IDENTITY}
    if cfg.use_image:
        for _k in [f"observation.images.{c}" for c in cfg.camera_names]:
            feats[_k] = PolicyFeature(type=FeatureType.VISUAL, shape=(3, 224, 224))
        norm["VISUAL"] = NormalizationMode.IDENTITY
    return _LRConfig(
        n_obs_steps=1, chunk_size=cfg.chunk_size, n_action_steps=cfg.chunk_size,
        input_features=feats,
        output_features={ACTION_KEY: PolicyFeature(type=FeatureType.ACTION,
                                                   shape=(cfg.action_dim,))},
        normalization_mapping=norm,
        paligemma_variant=cfg.paligemma_variant,
        action_expert_variant=cfg.action_expert_variant,
        max_state_dim=cfg.max_state_dim, max_action_dim=cfg.max_action_dim,
        num_inference_steps=cfg.num_inference_steps,
        tokenizer_max_length=cfg.tokenizer_max_length,
        dtype=cfg.dtype, gradient_checkpointing=cfg.gradient_checkpointing,
        freeze_vision_encoder=cfg.freeze_vision_encoder,
        # with LoRA the base VLM must be frozen — adapters carry the VLM update
        train_expert_only=cfg.train_expert_only or cfg.vlm_lora_rank > 0)


def _load_pretrained_weights(policy, repo_id):
    """Load openpi-ported weights with version-proof key remapping + a HARD check.

    lerobot's own from_pretrained loads with strict=False and only prints missing
    keys — under transformers >= 5.4 (which dropped the `.vision_model` nesting
    inside SigLIP) that silently leaves the ENTIRE vision tower random-init."""
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file

    sd = load_file(hf_hub_download(repo_id, "model.safetensors"))
    sd = policy._fix_pytorch_state_dict_keys(sd, policy.config)
    sd = {(k if k.startswith("model.") else f"model.{k}"): v for k, v in sd.items()}
    model_keys = set(policy.state_dict().keys())
    if (any(".vision_tower.vision_model." in k for k in sd)
            and not any(".vision_tower.vision_model." in k for k in model_keys)):
        sd = {k.replace(".vision_tower.vision_model.", ".vision_tower."): v
              for k, v in sd.items()}
    missing, unexpected = policy.load_state_dict(sd, strict=False)
    n_loaded = len(model_keys) - len(missing)
    print(f"pretrained load: {n_loaded}/{len(model_keys)} tensors from {repo_id} "
          f"({len(unexpected)} unexpected ignored)")
    if n_loaded < 0.99 * len(model_keys):
        raise RuntimeError(
            f"only {n_loaded}/{len(model_keys)} tensors matched {repo_id} — a partial "
            f"load silently finetunes random weights. First missing: {sorted(missing)[:5]}")


def _inject_vlm_lora(policy, rank, alpha, dropout, targets=None):
    """LoRA adapters on the (frozen) VLM's attention projections, in place —
    peft's inject_adapter_in_model keeps lerobot's module paths intact.
    Adapter params stay fp32 for stable AdamW on bf16 base weights."""
    from peft import LoraConfig, inject_adapter_in_model

    inject_adapter_in_model(
        LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=dropout,
                   target_modules=list(targets or ("q_proj", "k_proj", "v_proj", "o_proj")),
                   bias="none"),
        policy.model.paligemma_with_expert.paligemma)
    n_lora = 0
    for n, p in policy.named_parameters():
        if "lora_" in n:
            p.data = p.data.float(); p.requires_grad_(True); n_lora += p.numel()
    n_train = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    print(f"LoRA r={rank} on {len(targets or [1]*4)} VLM Linear types {list(targets) if targets else 'q/k/v/o'}: {n_lora/1e6:.1f}M adapter params; "
          f"total trainable {n_train/1e6:.0f}M (frozen base VLM + full action expert)")


@contextmanager
def build_context(device, dtype=None):
    """Construct the policy directly on `device` — never staged through host RAM.

    lerobot builds the policy wherever torch's default device points (CPU), then
    `.to(config.device)` at the end. For pi0.5 that means ~3.5B params materialise as
    fp32 in HOST RAM first — ~14 GB of allocation churn that the OOM-killer ends
    ("the kernel appears to have died") on a container whose cgroup memory limit is
    well under the host's RAM, long before the GPU is touched. Pointing torch's default
    device at the GPU for the duration of __init__ makes every nn.Linear allocate
    straight into VRAM: peak host RAM stays near zero, and the trailing .to() is a no-op.

    Passing `dtype` also redirects torch's default dtype, halving the build's peak VRAM
    (~14 GB fp32 -> ~7 GB bf16). Off by default because it is not numerically free:
    anything computed at construction lands in bf16 rather than fp32. Parameters do not
    care (the pretrained load overwrites every one), but a buffer derived at __init__ —
    RoPE inverse frequencies being the classic case — would keep the reduced precision.
    transformers guards inv_freq with an explicit .float(), so bf16 is believed safe;
    pass dtype="bfloat16" below only if the fp32 build genuinely does not fit."""
    want_cuda = str(device).startswith("cuda") and torch.cuda.is_available()
    if not want_cuda:
        yield
        return
    prev = torch.get_default_dtype()
    if dtype is not None:
        torch.set_default_dtype({"bfloat16": torch.bfloat16, "float16": torch.float16,
                                 "float32": torch.float32}[str(dtype).lower()])
    try:
        with torch.device(device):
            yield
    finally:
        torch.set_default_dtype(prev)


def pick_build_dtype(cfg_dtype, device, tight_gb=24.0):
    """Decide whether to construct in reduced precision, from FREE VRAM.

    Constructing pi0/pi05 in fp32 costs ~21 GB transiently before lerobot casts to
    cfg.dtype. That is fine on an empty 40 GB+ card and is numerically the safest
    path, so it stays the default. But free VRAM is often far below total: a crashed
    kernel keeps its CUDA context and allocations alive, and inside a container those
    processes usually cannot be signalled (nvidia-smi reports host PIDs you have no
    permission over). Rather than die with a CUDA OOM that reads as "model too big
    for this GPU", drop the build to cfg.dtype (~7 GB) when the headroom isn't there.

    Safe because every parameter is overwritten by the pretrained load moments later;
    only construction-time buffers keep the reduced precision, and transformers guards
    the one that matters (RoPE inv_freq) with an explicit .float()."""
    if not (str(device).startswith("cuda") and torch.cuda.is_available()):
        return None
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    if free_gb >= tight_gb:
        return None
    print(f"only {free_gb:.1f} GB VRAM free (< {tight_gb:.0f} GB) — constructing in "
          f"{cfg_dtype} (~7 GB) instead of fp32 (~21 GB). If that's unexpected, a dead "
          f"process is holding VRAM; restarting the pod is the clean fix.")
    return cfg_dtype


def quantize_vlm(policy, mode, lora_rank=0, expert_only=False,
                 compute_dtype=torch.bfloat16):
    """QLoRA-style k-bit quantization of the FROZEN VLM, in place. No-op when off.

    Swaps every nn.Linear inside the PaliGemma tower for a bitsandbytes Linear4bit
    (NF4 + double quantization) or Linear8bitLt, moving each to the GPU IMMEDIATELY —
    bitsandbytes quantizes lazily on .cuda(), so going layer-by-layer frees each
    full-precision weight as we go and keeps the peak at one layer instead of a
    second copy of the model.

    NF4 = 4-bit normal-float storage with bf16 dequantization at matmul time: the 2B
    VLM drops ~4.6 GB -> ~1.4 GB, which is what buys back the headroom for a larger
    batch. Throughput stays close to bf16; the cost is a small quality hit that the
    LoRA adapters largely absorb.

    Deliberately NOT quantized:
      • the action expert — it is FULLY trained, and 4-bit weights take no gradient,
        so quantizing it would silently freeze the one part that must learn;
      • lm_head / embeddings — vocabulary-sized and weight-tied;
      • norms — 1-D, negligible memory, and most damaged by quantization."""
    mode = (mode or "none").lower()
    if mode in ("none", "off", ""):
        return
    assert mode in ("nf4", "int8"), f"QUANTIZE must be none|nf4|int8, got {mode!r}"
    if lora_rank <= 0 and not expert_only:
        raise ValueError(
            f"quantize={mode!r} freezes the VLM (4-bit weights take no gradient) but "
            f"vlm_lora_rank=0 and train_expert_only=False — nothing would train.")
    import bitsandbytes as bnb

    vlm  = policy.model.paligemma_with_expert.paligemma
    skip = ("lm_head", "embed_tokens", "embed_out")
    n_swapped = saved = 0

    def _swap(module, prefix=""):
        nonlocal n_swapped, saved
        for name, child in list(module.named_children()):
            path = f"{prefix}.{name}" if prefix else name
            if isinstance(child, nn.Linear) and not any(s in path for s in skip):
                w = child.weight.data
                b = child.bias.data if child.bias is not None else None
                if mode == "nf4":
                    new = bnb.nn.Linear4bit(
                        child.in_features, child.out_features, bias=b is not None,
                        compute_dtype=compute_dtype, quant_type="nf4",
                        compress_statistics=True)          # double quantization
                    new.weight = bnb.nn.Params4bit(
                        w.to(compute_dtype), requires_grad=False,
                        quant_type="nf4", compress_statistics=True)
                else:
                    new = bnb.nn.Linear8bitLt(
                        child.in_features, child.out_features, bias=b is not None,
                        has_fp16_weights=False, threshold=6.0)
                    new.weight = bnb.nn.Int8Params(
                        w.to(torch.float16), requires_grad=False, has_fp16_weights=False)
                if b is not None:
                    new.bias = nn.Parameter(b.to(compute_dtype), requires_grad=False)
                # move NOW — this is where bnb quantizes, and it lets the
                # full-precision weight be freed before the next layer is built.
                setattr(module, name, new.to("cuda"))
                n_swapped += 1
                saved += w.numel() * (w.element_size() - (0.5 if mode == "nf4" else 1))
                del w, b, child
            else:
                _swap(child, path)

    _swap(vlm)
    torch.cuda.empty_cache()

    # QLoRA + gradient checkpointing: the checkpointed blocks sit behind a fully frozen
    # base, so without an input that requires grad the recomputed graph is detached and
    # the adapters get NO gradient — training silently does nothing.
    if getattr(policy.config, "gradient_checkpointing", False):
        try:
            vlm.enable_input_require_grads()
        except AttributeError:
            print("WARNING: could not enable input grads on the VLM; if LoRA grads "
                  "come back None, set gradient_checkpointing False")

    print(f"quantized VLM to {mode.upper()}: {n_swapped} Linear layers, "
          f"~{saved/1e9:.1f} GB saved (action expert + lm_head left in bf16)")

class PiPolicy(nn.Module):
    """π0 wrapper — mean-std norm in the wrapper, IDENTITY inside lerobot."""

    def __init__(self, cfg: PolicyCfg, stats: dict, device=None):
        super().__init__()
        self.cfg = cfg
        self._quantized = str(getattr(cfg, 'quantize', 'none')).lower() in ('nf4', 'int8')
        self.image_keys = [f"observation.images.{c}" for c in cfg.camera_names]
        # Order is load -> quantize -> LoRA, and it is not interchangeable:
        # quantization is destructive (NF4-ing random weights and then loading over
        # Params4bit does not round-trip), and peft must see Linear4bit to build
        # lora.Linear4bit wrappers.
        with build_context(device, pick_build_dtype(cfg.dtype, device)):
            self.policy = PI0Policy(_lerobot_config(cfg))
        if cfg.pretrained:
            _load_pretrained_weights(self.policy, cfg.pretrained)
        else:
            print("WARNING: random-init weights — smoke tests only, NOT finetuning")
        quantize_vlm(self.policy, cfg.quantize, cfg.vlm_lora_rank, cfg.train_expert_only)
        if cfg.vlm_lora_rank > 0:
            _inject_vlm_lora(self.policy, cfg.vlm_lora_rank,
                             cfg.vlm_lora_alpha, cfg.vlm_lora_dropout,
                             targets=cfg.vlm_lora_targets)
        self.tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")
        for k in ("state_mean", "state_std", "action_mean", "action_std"):
            self.register_buffer(k, torch.as_tensor(stats[k]).float())
        self.register_buffer("_imagenet_mean", _IMN_MEAN.clone())
        self.register_buffer("_imagenet_std",  _IMN_STD.clone())

    def _norm_state(self, s):    return (s - self.state_mean) / self.state_std
    def _norm_action(self, a):   return (a - self.action_mean) / self.action_std
    def _unnorm_action(self, a): return a * self.action_std + self.action_mean
    def _to_raw(self, img):      # undo ImageNet norm -> [0,1]; lerobot maps to [-1,1]
        return (img * self._imagenet_std + self._imagenet_mean).clamp(0, 1)

    def _make_batch(self, obs_state, actions=None, action_is_pad=None,
                    obs_image=None, task=None, training=None):
        if training is None:
            training = self.training
        B = obs_state.shape[0]
        task = task or [TASK_TEXT] * B
        state = mask_state(self._norm_state(obs_state), self.cfg.proprio_mode,
                           self.cfg.proprio_dropout_rate, training)
        batch = {STATE_KEY: state}                     # (B, state_dim) — no seq dim
        if self.cfg.use_image and obs_image is not None:
            if torch.is_tensor(obs_image):                 # 1 cam -> dict
                obs_image = {self.image_keys[0]: obs_image}
            for _k in self.image_keys:                     # feed each camera
                batch[_k] = self._to_raw(obs_image[_k])
        enc = self.tokenizer(list(task), return_tensors="pt", padding="max_length",
                             truncation=True, max_length=self.cfg.tokenizer_max_length)
        batch[LANG_TOKENS]    = enc["input_ids"].to(obs_state.device)
        batch[LANG_ATTN_MASK] = enc["attention_mask"].to(obs_state.device)
        if actions is not None:
            batch[ACTION_KEY]      = self._norm_action(actions)
            batch["action_is_pad"] = action_is_pad
        return batch

    def _amp(self):
        # QLoRA keeps LoRA adapters fp32; without autocast they upcast activations
        # and collide with the bf16 (unquantized) action expert -> 'mat1 float !=
        # mat2 BFloat16'. Autocast the forward to bf16 when the VLM is k-bit quantized.
        from contextlib import nullcontext
        if self._quantized and torch.cuda.is_available():
            return torch.autocast('cuda', dtype=torch.bfloat16)
        return nullcontext()

    def forward(self, obs_state, actions, action_is_pad, obs_image=None, task=None):
        with self._amp():
            loss, _ = self.policy.forward(
                self._make_batch(obs_state, actions, action_is_pad, obs_image, task))
        return loss, loss.item(), 0.0

    def reset(self):
        self.policy.reset()

    @torch.no_grad()
    def predict(self, obs_state, obs_image=None, task=None):
        with self._amp():
            a = self.policy.select_action(
                self._make_batch(obs_state, obs_image=obs_image, task=task, training=False))
        return self._unnorm_action(a)


def build_model(cfg: dict, stats: dict, device):
    m, d = cfg["model"], cfg["dataset"]
    return PiPolicy(PolicyCfg(
        state_dim=m["state_dim"], action_dim=m["action_dim"],
        chunk_size=d["chunk_size"], use_image=d["use_image"],
        camera_names=tuple(d.get("camera_names", ("wrist_cam", "scene_cam"))),
        tokenizer_max_length=m["tokenizer_max_length"],
        pretrained=m.get("pretrained", ""),
        vlm_lora_rank=m.get("vlm_lora_rank", 0),
        vlm_lora_alpha=m.get("vlm_lora_alpha", 32),
        vlm_lora_dropout=m.get("vlm_lora_dropout", 0.05),
        vlm_lora_targets=tuple(m.get("vlm_lora_targets",
                                     ("q_proj", "k_proj", "v_proj", "o_proj"))),
        dtype=m["dtype"], gradient_checkpointing=m["gradient_checkpointing"],
        freeze_vision_encoder=m["freeze_vision_encoder"],
        train_expert_only=m["train_expert_only"],
        quantize=m.get("quantize", "none") or "none",
        proprio_mode=m["proprio_mode"],
        proprio_dropout_rate=m["proprio_dropout_rate"]),
        stats, device=device).to(device)

print("π0 wrapper defined")

## 4 · Load checkpoint + reconstruct the val split

In [ ]:
import torch, json, pathlib, random, collections

ck_path = pathlib.Path(CKPT_DIR) / "best.pt"
assert ck_path.exists(), f"no best.pt in {CKPT_DIR} yet — wait for the first epoch"
print(f"loading {ck_path} ...")
ck = torch.load(ck_path, map_location="cpu", weights_only=False)
CFG = ck["config"]
CFG["model"]["pretrained"] = ""          # weights come from the checkpoint, not the base
CHUNK_SIZE = CFG["dataset"]["chunk_size"]
SEED       = CFG["training"]["seed"]
VAL_FRAC   = CFG["dataset"]["val_frac"]
print(f"checkpoint: policy={ck['policy']} epoch={ck['epoch']} val_l1={ck['val_l1']:.4f}")

model = build_model(CFG, ck["stats"], torch.device("cuda"))
missing, unexpected = model.load_state_dict(ck["model_state"], strict=True), None
model.eval()
del ck["model_state"]
import gc; gc.collect(); torch.cuda.empty_cache()

# reconstruct the EXACT training val split (stratified when canonical_tasks.json exists)
n_eps = json.loads((pathlib.Path(DATA_DIR) / "meta/info.json").read_text())["total_episodes"]
_cpath = pathlib.Path(DATA_DIR) / "meta/canonical_tasks.json"
if _cpath.exists():
    _canon = {int(k): v["canonical"] for k, v in json.loads(_cpath.read_text()).items()}
    _groups = collections.defaultdict(list)
    for _e in range(n_eps):
        _groups[_canon.get(_e, "?")].append(_e)
    _rng = random.Random(SEED)
    val_eps = []
    for _t in sorted(_groups):
        _g = sorted(_groups[_t]); _rng.shuffle(_g)
        val_eps += _g[:max(1, round(VAL_FRAC * len(_g)))]
    val_eps = sorted(val_eps)
else:
    _, val_eps = FR5Dataset.episode_split(n_eps, VAL_FRAC, SEED)
print(f"val split reconstructed: {len(val_eps)} episodes (must match training: 18)")

## 5 · Language probe + scorecard

In [ ]:
# ── 12e · language-grounding probe (offline, ~3-5 min, no robot) ──────────────
# Answers ONE question per checkpoint: does the instruction change the actions?
# For each probed val episode, three rollouts over the same frames:
#   A: TRUE instruction        B: TRUE again (fresh noise -> the NOISE FLOOR
#   the robot-side A/B lacked) W: WRONG instruction (different canonical task)
# Metrics:  lang_ratio = |A-W| / |A-B|  ->  ~1.0 = language IGNORED (wrong
# instruction moves actions no more than re-sampling does); >~2 = grounded.
# Also tracks grasp intent (max gripper channel) — the other missing behavior.

import json, pathlib
import numpy as np, torch

_canon = json.loads((pathlib.Path(DATA_DIR) / "meta/canonical_tasks.json").read_text())

# pick PROBE_EPISODES val episodes with DISTINCT canonical tasks
_seen, _chosen = set(), []
for _e in val_eps:
    _c = _canon[str(_e)]["canonical"]
    if _c not in _seen:
        _seen.add(_c); _chosen.append(int(_e))
    if len(_chosen) == PROBE_EPISODES:
        break

def _wrong_instruction(ep):
    """The varied instruction of a val episode from a DIFFERENT canonical task."""
    own = _canon[str(ep)]["canonical"]
    for _e in val_eps:
        if _canon[str(_e)]["canonical"] != own:
            return _canon[str(_e)]["instruction"], _canon[str(_e)]["canonical"]
    raise RuntimeError("no other canonical task in val set")

def _rollout(ep, instruction, n):
    ds = FR5Dataset(DATA_DIR, chunk_size=CHUNK_SIZE, episode_indices=[ep],
                    aug_level="none", frame_stride=1)   # sequential: queue stays in sync
    n = min(n, len(ds))
    model.eval(); model.reset()
    preds, gts = [], []
    with torch.no_grad():
        for i in range(n):
            it = ds[i]
            a = model.predict(it["observation.state"].unsqueeze(0).cuda(),
                              obs_image={k: it[k].unsqueeze(0).cuda() for k in ds.camera_keys},
                              task=[instruction])
            preds.append(a.squeeze(0).float().cpu().numpy())
            gts.append(it["action"][0].numpy())
    return np.asarray(preds), np.asarray(gts)

print(f"probing {len(_chosen)} val episodes x 3 rollouts x {PROBE_FRAMES} frames "
      f"(expect benign imread warnings — off-grid frames decode from video)\n")
_rows = []
for _ep in _chosen:
    _true = _canon[str(_ep)]["instruction"]
    _wrong, _wc = _wrong_instruction(_ep)
    A, gt = _rollout(_ep, _true, PROBE_FRAMES)
    B, _  = _rollout(_ep, _true, PROBE_FRAMES)      # noise-floor control
    W, _  = _rollout(_ep, _wrong, PROBE_FRAMES)
    floor = float(np.abs(A[:, :6] - B[:, :6]).mean())
    lang  = float(np.abs(A[:, :6] - W[:, :6]).mean())
    ratio = lang / max(floor, 1e-6)
    mae_t = float(np.abs(A - gt).mean()); mae_w = float(np.abs(W - gt).mean())
    _rows.append((_ep, ratio, floor, lang, mae_t, mae_w, float(A[:, 6].max())))
    print(f"ep{_ep:03d}  '{_canon[str(_ep)]['canonical'][:38]}'")
    print(f"   vs wrong: '{_wc[:38]}'")
    print(f"   noise floor |A-B| {floor:.3f} deg   language |A-W| {lang:.3f} deg   "
          f"ratio {ratio:.2f}")
    print(f"   MAE true {mae_t:.3f}  MAE wrong {mae_w:.3f}   max gripper {A[:,6].max():+.2f}\n")

# plan-consistency from the TRUE rollouts: chunk-boundary jumps
_alljumps = []
for _ep in _chosen:
    pass  # boundary jumps computed below from stored rollouts
_r = np.array([r[1] for r in _rows])
_g = max(r[6] for r in _rows)
print("=" * 66)
print(f"language sensitivity ratio: mean {_r.mean():.2f}  (per-ep {np.round(_r,2).tolist()})")
print(f"   ~1.0 -> language IGNORED (wrong instruction ~= re-sample noise)")
print(f"   >2   -> instructions measurably steer the policy")
print(f"grasp intent: max gripper channel {_g:+.2f}  (close threshold 0.65; "
      f"{'NO grasp intent yet' if _g < 0.35 else 'grasp intent emerging'})")
print("track BOTH numbers per checkpoint — they are the two capabilities the "
      "robot runs showed missing.")


## 6 · Plan-consistency spot check

In [ ]:
# plan-consistency metric from the same probe rollouts (no extra compute):
# re-derive boundary jumps by one more TRUE rollout per episode would double cost —
# instead run it on the FIRST probed episode only, as a spot check.
import numpy as np
_ep = _chosen[0]
A, _ = _rollout(_ep, _canon[str(_ep)]["instruction"], PROBE_FRAMES)
_jump = np.abs(np.diff(A[:, :6], axis=0)).max(1)
_bounds = [_jump[i-1] for i in range(50, len(A), 50)]
print(f"plan consistency (ep{_ep:03d}): boundary jumps {np.round(_bounds,1).tolist()} deg "
      f"(2-epoch baseline median 6.7; converged target ~1)")